In [ ]:
import contextlib

import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from jaxtyping import Float
from plotly.subplots import make_subplots
from torch import Tensor
from tqdm.autonotebook import tqdm

from src.config.base import BaseConfig
from src.method.drifting.kernel import (
    compute_gaussian_drifting_statistics,
    compute_gaussian_drifting_statistics_at_query,
)
from src.model.mlp import StackedResidualMLPConfig

DEVICE = (
    'cuda:1'
    if torch.cuda.is_available() and torch.cuda.device_count() > 1
    else ('cuda:0' if torch.cuda.is_available() else 'cpu')
)
USE_CUDA = DEVICE.startswith('cuda')
AMP_DTYPE = torch.bfloat16

torch.set_float32_matmul_precision('high')


@contextlib.contextmanager
def training_device_context():
    if USE_CUDA:
        with (
            torch.cuda.device(DEVICE),
            torch.device(DEVICE),
            torch.autocast(device_type='cuda', dtype=AMP_DTYPE),
        ):
            yield
        return
    with torch.device(DEVICE):
        yield


print(f'device: {DEVICE}')


In [ ]:
class BimodalDataConfig(BaseConfig):
    offset: Float[Tensor, '2']
    center: float
    std: float
    scale: float
    std_cap: float = 2

    def _sample_capped_noise(self, batch_size: int) -> Float[Tensor, 'batch 2']:
        z = torch.randn((batch_size, 2))
        norms = z.norm(dim=1, keepdim=True).clamp_min(1e-6)
        capped_norms = norms.clamp_max(self.std_cap)
        return z * (self.std * capped_norms / norms)

    def sample_mode(
        self,
        batch_size: int,
        mode_id: int,
    ) -> Float[Tensor, 'batch 2']:
        x = self._sample_capped_noise(batch_size)
        direction = 1.0 if mode_id == 0 else -1.0
        x[:, 0] += direction * self.center / self.scale
        return x * self.scale + self.offset

    def sample_with_mode_ids(
        self,
        batch_size: int,
    ) -> tuple[Float[Tensor, 'batch 2'], Tensor]:
        if batch_size % 2 != 0:
            raise ValueError('batch_size must be even')
        per_mode_size = batch_size // 2
        mode_ids = torch.cat(
            [
                torch.zeros(per_mode_size, dtype=torch.long),
                torch.ones(per_mode_size, dtype=torch.long),
            ]
        )
        x = torch.cat(
            [
                self.sample_mode(per_mode_size, mode_id=0),
                self.sample_mode(per_mode_size, mode_id=1),
            ],
            dim=0,
        )
        return x, mode_ids

    def sample(self, batch_size: int) -> Float[Tensor, 'batch 2']:
        x, _ = self.sample_with_mode_ids(batch_size)
        return x

    def sample_prior(self, batch_size: int) -> Float[Tensor, 'batch 2']:
        return torch.randn((batch_size, 2))

    def sample_contours(
        self,
        per_contour_size: int,
        zscores: tuple[float, ...] = (0.1, 0.5, 1.0, 1.5),
    ) -> tuple[Float[Tensor, 'batch 2'], Tensor, tuple[float, ...]]:
        angles = torch.linspace(0.0, 2.0 * torch.pi, per_contour_size + 1)[:-1]
        unit_circle = torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)
        contour_points = []
        contour_ids = []
        for contour_id, zscore in enumerate(zscores):
            contour_points.append(zscore * unit_circle)
            contour_ids.append(
                torch.full((per_contour_size,), contour_id, dtype=torch.long)
            )
        return torch.cat(contour_points, dim=0), torch.cat(contour_ids, dim=0), zscores


class DriftingComparisonConfig(BaseConfig):
    objective: str
    train_batch_size: int
    total_steps: int
    lr: float
    weight_decay: float
    grad_clip_norm: float
    bandwidth: float
    transport_step_size: float
    max_transport_update_norm: float
    inverse_cycle_weight: float
    log_every_steps: int
    snapshot_every_steps: int
    scatter_num_samples: int
    contour_points_per_level: int
    contour_grid_resolution: int
    vector_grid_resolution: int
    contour_levels: tuple[float, ...]


data_config = BimodalDataConfig(
    offset=torch.tensor([0.0, 0.0]),
    center=3.0,
    std=0.2,
    scale=2.0,
)

experiment_config = DriftingComparisonConfig(
    objective='reverse_kl',
    train_batch_size=2048,
    total_steps=150000,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip_norm=1.0,
    bandwidth=0.75,
    transport_step_size=0.08,
    max_transport_update_norm=0.75,
    inverse_cycle_weight=0.5,
    log_every_steps=50,
    snapshot_every_steps=20000,
    scatter_num_samples=4096,
    contour_points_per_level=160,
    contour_grid_resolution=144,
    vector_grid_resolution=30,
    contour_levels=(0.1, 0.5, 1.0, 1.5),
)

experiment_config


In [ ]:
class NaiveDriftingComparisonModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.decoder = StackedResidualMLPConfig.initialize(
            layer_dims=[2, 512, 512, 512, 512, 2],
        ).get_model()
        self.inverse = StackedResidualMLPConfig.initialize(
            layer_dims=[2, 512, 512, 512, 512, 2],
        ).get_model()

    def detached_state(self, module: nn.Module) -> dict[str, Tensor]:
        return {
            **{
                name: parameter.detach()
                for name, parameter in module.named_parameters()
            },
            **{name: buffer.detach() for name, buffer in module.named_buffers()},
        }

    def decode(self, z: Float[Tensor, 'batch 2']) -> Float[Tensor, 'batch 2']:
        return self.decoder(z)

    def encode(self, x: Float[Tensor, 'batch 2']) -> Float[Tensor, 'batch 2']:
        return self.inverse(x)

    def frozen_decode(
        self,
        y: Float[Tensor, 'batch 2'],
    ) -> Float[Tensor, 'batch 2']:
        return torch.func.functional_call(
            self.decoder,
            self.detached_state(self.decoder),
            args=(y,),
        )

    def get_optimizers(
        self,
    ) -> tuple[torch.optim.Optimizer, torch.optim.Optimizer]:
        return (
            torch.optim.AdamW(
                self.decoder.parameters(),
                lr=experiment_config.lr,
                weight_decay=experiment_config.weight_decay,
            ),
            torch.optim.AdamW(
                self.inverse.parameters(),
                lr=experiment_config.lr,
                weight_decay=experiment_config.weight_decay,
            ),
        )


model = NaiveDriftingComparisonModel().to(DEVICE)
decoder_op, inverse_op = model.get_optimizers()

logs = {
    'steps': [],
    'losses': {
        'drift_matching': [],
        'inverse_latent': [],
        'inverse_cycle': [],
    },
    'metrics': {
        'drift_norm': [],
        'data_density': [],
        'model_density': [],
        'inverse_mode_gap': [],
    },
}


In [ ]:
MODE_COLORS = ('#1f77b4', '#ff7f0e')
CONTOUR_COLORS = ('#440154', '#3b528b', '#21918c', '#5ec962', '#fde725')
LOSS_COLORS = {
    'drift_matching': '#d62728',
    'inverse_latent': '#1f77b4',
    'inverse_cycle': '#ff7f0e',
}
METRIC_COLORS = {
    'drift_norm': '#2ca02c',
    'data_density': '#9467bd',
    'model_density': '#8c564b',
    'inverse_mode_gap': '#e377c2',
}


def current_training_step() -> int:
    if len(logs['steps']) == 0:
        return 0
    return int(logs['steps'][-1])


def trailing_window(
    values: list[float],
    window_size: int = 2000,
) -> tuple[list[int], list[float]]:
    start = max(0, len(values) - window_size)
    return logs['steps'][start:], values[start:]


def subsample_indices(total_size: int, max_points: int) -> Tensor:
    if total_size <= max_points:
        return torch.arange(total_size)
    return torch.linspace(0, total_size - 1, max_points).round().long()


def normalize_vectors(
    vectors: Float[Tensor, 'batch 2'],
    display_length: float,
) -> tuple[Float[Tensor, 'batch 2'], Float[Tensor, 'batch']]:
    magnitudes = vectors.norm(dim=1)
    normalized = vectors / magnitudes.unsqueeze(-1).clamp_min(1e-6)
    return display_length * normalized, magnitudes


def square_grid_limits(
    points: Float[Tensor, 'batch 2'],
    padding: float = 0.12,
) -> tuple[float, float, float, float]:
    mins = points.min(dim=0).values
    maxs = points.max(dim=0).values
    center = 0.5 * (mins + maxs)
    radius = 0.5 * float((maxs - mins).max().item())
    radius = max(radius * (1.0 + padding), 1.0)
    return (
        float(center[0] - radius),
        float(center[0] + radius),
        float(center[1] - radius),
        float(center[1] + radius),
    )


def build_square_grid(
    points: Float[Tensor, 'batch 2'],
    resolution: int,
) -> tuple[Float[Tensor, 'grid 2'], Tensor, Tensor]:
    x_min, x_max, y_min, y_max = square_grid_limits(points)
    xs = torch.linspace(
        x_min,
        x_max,
        resolution,
        device=points.device,
        dtype=points.dtype,
    )
    ys = torch.linspace(
        y_min,
        y_max,
        resolution,
        device=points.device,
        dtype=points.dtype,
    )
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
    grid_points = torch.stack([grid_x.reshape(-1), grid_y.reshape(-1)], dim=-1)
    return grid_points, xs.detach().cpu(), ys.detach().cpu()


def regular_grid_display_length(
    xs: Tensor,
    ys: Tensor,
    fraction: float = 1.05,
) -> float:
    if xs.numel() < 2 or ys.numel() < 2:
        return fraction
    x_step = float((xs[1] - xs[0]).item())
    y_step = float((ys[1] - ys[0]).item())
    return fraction * min(abs(x_step), abs(y_step))


def add_quiver_traces(
    fig: go.Figure,
    *,
    points: Float[Tensor, 'batch 2'],
    vectors: Float[Tensor, 'batch 2'],
    magnitudes: Float[Tensor, 'batch'],
    color: str,
    name: str,
    row: int,
    col: int,
) -> None:
    quiver = ff.create_quiver(
        x=points[:, 0].detach().cpu().float().tolist(),
        y=points[:, 1].detach().cpu().float().tolist(),
        u=vectors[:, 0].detach().cpu().float().tolist(),
        v=vectors[:, 1].detach().cpu().float().tolist(),
        scale=1.0,
        arrow_scale=0.42,
        line_color=color,
        name=name,
    )
    for trace_index, trace in enumerate(quiver.data):
        trace.name = name
        trace.showlegend = trace_index == 0
        trace.line.width = 1.6
        fig.add_trace(trace, row=row, col=col)

    hover_text = [
        f'{name}<br>|v|={float(magnitude):.4f}'
        for magnitude in magnitudes.detach().cpu().float().tolist()
    ]
    fig.add_trace(
        go.Scatter(
            x=points[:, 0].detach().cpu().float().tolist(),
            y=points[:, 1].detach().cpu().float().tolist(),
            mode='markers',
            marker={'size': 11, 'color': color, 'opacity': 0.001},
            text=hover_text,
            hovertemplate='%{text}<extra></extra>',
            showlegend=False,
            name=name,
        ),
        row=row,
        col=col,
    )


def plot_training_losses() -> go.Figure:
    step = current_training_step()
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=['optimization losses', 'drift and inverse diagnostics'],
        horizontal_spacing=0.08,
    )
    for loss_name, loss_values in logs['losses'].items():
        xs, ys = trailing_window(loss_values)
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode='lines',
                line={'width': 3, 'color': LOSS_COLORS[loss_name]},
                name=loss_name,
            ),
            row=1,
            col=1,
        )
    for metric_name, metric_values in logs['metrics'].items():
        xs, ys = trailing_window(metric_values)
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode='lines',
                line={'width': 3, 'color': METRIC_COLORS[metric_name]},
                name=metric_name,
            ),
            row=1,
            col=2,
        )
    fig.update_layout(
        title=f'naive drifting comparison losses at step {step}',
        height=360,
        width=1450,
        margin={'l': 50, 'r': 20, 't': 75, 'b': 40},
        legend={'x': 1.02, 'xanchor': 'left', 'y': 1.0, 'yanchor': 'top'},
    )
    fig.update_xaxes(title_text='optimization step', showgrid=True, zeroline=False)
    fig.update_yaxes(showgrid=True, zeroline=False)
    return fig


def plot_snapshot(scatter_num_samples: int) -> go.Figure:
    step = current_training_step()
    model_was_training = model.training
    model.eval()
    contour_points, contour_ids, contour_levels = data_config.sample_contours(
        per_contour_size=experiment_config.contour_points_per_level,
        zscores=experiment_config.contour_levels,
    )
    with torch.no_grad():
        x_data, data_mode_ids = data_config.sample_with_mode_ids(scatter_num_samples)
        x_data = x_data.to(DEVICE)
        z_prior = data_config.sample_prior(scatter_num_samples).to(DEVICE)
        x_model = model.decode(z_prior)
        x_contours = model.decode(contour_points.to(DEVICE))
        y_data = model.encode(x_data)

        sample_space_points = torch.cat([x_data, x_model], dim=0)
        contour_grid_points, contour_xs, contour_ys = build_square_grid(
            sample_space_points,
            resolution=experiment_config.contour_grid_resolution,
        )
        contour_statistics = compute_gaussian_drifting_statistics_at_query(
            query_samples=contour_grid_points.float(),
            model_reference_samples=x_model.float(),
            data_samples=x_data.float(),
            bandwidth=experiment_config.bandwidth,
            objective=experiment_config.objective,
            stability_eps=1e-4,
            exclude_model_reference_diagonal=False,
        )
        drift_norm = contour_statistics.drift.norm(dim=1).reshape(
            experiment_config.contour_grid_resolution,
            experiment_config.contour_grid_resolution,
        )

        vector_grid_points, vector_xs, vector_ys = build_square_grid(
            sample_space_points,
            resolution=experiment_config.vector_grid_resolution,
        )
        vector_statistics = compute_gaussian_drifting_statistics_at_query(
            query_samples=vector_grid_points.float(),
            model_reference_samples=x_model.float(),
            data_samples=x_data.float(),
            bandwidth=experiment_config.bandwidth,
            objective=experiment_config.objective,
            stability_eps=1e-4,
            exclude_model_reference_diagonal=False,
        )
        vector_display_length = regular_grid_display_length(vector_xs, vector_ys)
        display_vectors, vector_magnitudes = normalize_vectors(
            vector_statistics.drift.detach().cpu(),
            vector_display_length,
        )

    if model_was_training:
        model.train()

    fig = make_subplots(
        rows=1,
        cols=4,
        subplot_titles=[
            'data and decoder(prior)',
            'prior contours pushed through decoder',
            'direct KDE drifting field',
            'diagnostic inverse embedding',
        ],
        horizontal_spacing=0.05,
    )
    fig.add_trace(
        go.Scatter(
            x=x_data[:, 0].detach().cpu().numpy(),
            y=x_data[:, 1].detach().cpu().numpy(),
            mode='markers',
            marker={'size': 4, 'color': '#202020', 'opacity': 0.18},
            name='data',
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=x_model[:, 0].detach().cpu().numpy(),
            y=x_model[:, 1].detach().cpu().numpy(),
            mode='markers',
            marker={'size': 4, 'color': '#d62728', 'opacity': 0.45},
            name='decoder(prior)',
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=x_data[:, 0].detach().cpu().numpy(),
            y=x_data[:, 1].detach().cpu().numpy(),
            mode='markers',
            marker={'size': 4, 'color': '#202020', 'opacity': 0.10},
            name='data backdrop',
        ),
        row=1,
        col=2,
    )
    for contour_id, contour_level in enumerate(contour_levels):
        mask = contour_ids == contour_id
        fig.add_trace(
            go.Scatter(
                x=x_contours[mask, 0].detach().cpu().numpy(),
                y=x_contours[mask, 1].detach().cpu().numpy(),
                mode='lines',
                line={
                    'width': 2.4,
                    'color': CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                },
                name=f'z = {contour_level}',
            ),
            row=1,
            col=2,
        )

    fig.add_trace(
        go.Contour(
            x=contour_xs.numpy(),
            y=contour_ys.numpy(),
            z=drift_norm.detach().cpu().numpy(),
            colorscale='Viridis',
            ncontours=18,
            contours={'coloring': 'heatmap', 'showlabels': False},
            line={'width': 0.8},
            opacity=0.38,
            showscale=False,
            hovertemplate='x=%{x:.2f}<br>y=%{y:.2f}<br>||drift||=%{z:.4f}<extra></extra>',
            name='drift norm',
        ),
        row=1,
        col=3,
    )
    fig.add_trace(
        go.Scatter(
            x=x_data[:, 0].detach().cpu().numpy(),
            y=x_data[:, 1].detach().cpu().numpy(),
            mode='markers',
            marker={'size': 4, 'color': '#202020', 'opacity': 0.10},
            name='data field backdrop',
        ),
        row=1,
        col=3,
    )
    fig.add_trace(
        go.Scatter(
            x=x_model[:, 0].detach().cpu().numpy(),
            y=x_model[:, 1].detach().cpu().numpy(),
            mode='markers',
            marker={'size': 4, 'color': '#111111', 'opacity': 0.16},
            name='model field support',
        ),
        row=1,
        col=3,
    )
    add_quiver_traces(
        fig,
        points=vector_grid_points.detach().cpu(),
        vectors=display_vectors,
        magnitudes=vector_magnitudes,
        color='#d62728',
        name='direct drift',
        row=1,
        col=3,
    )

    latent_contours, latent_contour_ids, latent_contour_levels = data_config.sample_contours(
        per_contour_size=experiment_config.contour_points_per_level,
        zscores=experiment_config.contour_levels,
    )
    for contour_id, contour_level in enumerate(latent_contour_levels):
        mask = latent_contour_ids == contour_id
        fig.add_trace(
            go.Scatter(
                x=latent_contours[mask, 0].detach().cpu().numpy(),
                y=latent_contours[mask, 1].detach().cpu().numpy(),
                mode='lines',
                line={
                    'width': 1.6,
                    'color': CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                    'dash': 'dot',
                },
                name=f'latent z = {contour_level}',
            ),
            row=1,
            col=4,
        )
    for mode_id, color in enumerate(MODE_COLORS):
        mask = data_mode_ids == mode_id
        fig.add_trace(
            go.Scatter(
                x=y_data[mask, 0].detach().cpu().numpy(),
                y=y_data[mask, 1].detach().cpu().numpy(),
                mode='markers',
                marker={'size': 5, 'color': color, 'opacity': 0.55},
                name=f'inverse(data) mode {mode_id}',
            ),
            row=1,
            col=4,
        )

    for col in [1, 2, 3, 4]:
        fig.update_xaxes(showgrid=True, zeroline=False, row=1, col=col)
        fig.update_yaxes(showgrid=True, zeroline=False, row=1, col=col)
    for col, axis_name in [(1, 'y'), (2, 'y2'), (3, 'y3'), (4, 'y4')]:
        fig.update_xaxes(scaleanchor=axis_name, scaleratio=1, row=1, col=col)

    fig.update_layout(
        title=f'naive drifting comparison snapshot at step {step}',
        height=430,
        width=1850,
        margin={'l': 30, 'r': 20, 't': 85, 'b': 30},
        legend={'x': 1.02, 'xanchor': 'left', 'y': 1.0, 'yanchor': 'top'},
    )
    return fig


In [ ]:
def append_log_entry(
    step: int,
    drift_matching_loss: float,
    inverse_latent_loss: float,
    inverse_cycle_loss: float,
    drift_norm: float,
    data_density: float,
    model_density: float,
    inverse_mode_gap: float,
) -> None:
    logs['steps'].append(step)
    logs['losses']['drift_matching'].append(drift_matching_loss)
    logs['losses']['inverse_latent'].append(inverse_latent_loss)
    logs['losses']['inverse_cycle'].append(inverse_cycle_loss)
    logs['metrics']['drift_norm'].append(drift_norm)
    logs['metrics']['data_density'].append(data_density)
    logs['metrics']['model_density'].append(model_density)
    logs['metrics']['inverse_mode_gap'].append(inverse_mode_gap)


def snapshot_progress(
    scatter_num_samples: int = 1024,
    show: bool = True,
) -> dict[str, go.Figure]:
    figures = {
        'losses': plot_training_losses(),
        'snapshot': plot_snapshot(scatter_num_samples=scatter_num_samples),
    }
    if show:
        for figure in figures.values():
            figure.show()
    return figures


In [ ]:
for step in tqdm(range(1, experiment_config.total_steps + 1)):
    x_data, data_mode_ids = data_config.sample_with_mode_ids(
        experiment_config.train_batch_size,
    )
    x_data = x_data.to(DEVICE)
    data_mode_ids = data_mode_ids.to(DEVICE)
    z_prior = data_config.sample_prior(experiment_config.train_batch_size).to(DEVICE)

    decoder_op.zero_grad(set_to_none=True)
    with training_device_context():
        x_model = model.decode(z_prior)
        drifting_statistics = compute_gaussian_drifting_statistics(
            model_samples=x_model.detach().float(),
            data_samples=x_data.detach().float(),
            bandwidth=experiment_config.bandwidth,
            objective=experiment_config.objective,
            stability_eps=1e-4,
            exclude_self_interactions=True,
        )
        drift = drifting_statistics.drift.to(device=x_model.device, dtype=x_model.dtype)
        drift_norms = drift.norm(dim=1, keepdim=True).clamp_min(1e-6)
        clipped_drift = drift * torch.clamp(
            experiment_config.max_transport_update_norm / drift_norms,
            max=1.0,
        )
        x_target = (
            x_model.detach() + experiment_config.transport_step_size * clipped_drift.detach()
        ).detach()
        drift_matching_loss = F.huber_loss(
            x_model.float(),
            x_target.float(),
            delta=3.0,
        )
    drift_matching_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        model.decoder.parameters(),
        max_norm=experiment_config.grad_clip_norm,
    )
    decoder_op.step()

    inverse_op.zero_grad(set_to_none=True)
    with training_device_context():
        detached_model_samples = model.decode(z_prior).detach()
        recovered_latents = model.encode(detached_model_samples)
        inverse_latent_loss = F.huber_loss(
            recovered_latents.float(),
            z_prior.detach().float(),
            delta=2.0,
        )
        encoded_data = model.encode(x_data.detach())
        reconstructed_data = model.frozen_decode(encoded_data)
        inverse_cycle_loss = F.huber_loss(
            reconstructed_data.float(),
            x_data.detach().float(),
            delta=3.0,
        )
        inverse_loss = (
            inverse_latent_loss
            + experiment_config.inverse_cycle_weight * inverse_cycle_loss
        )
    inverse_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        model.inverse.parameters(),
        max_norm=experiment_config.grad_clip_norm,
    )
    inverse_op.step()

    with torch.no_grad():
        left_mean = encoded_data[data_mode_ids == 0].mean(dim=0)
        right_mean = encoded_data[data_mode_ids == 1].mean(dim=0)
        inverse_mode_gap = float((left_mean - right_mean).norm().item())
        drift_norm = float(drift.norm(dim=1).mean().item())
        data_density = float(drifting_statistics.data_density.mean().item())
        model_density = float(drifting_statistics.model_density.mean().item())

    should_log = (
        step == 1
        or step % experiment_config.log_every_steps == 0
        or step % experiment_config.snapshot_every_steps == 0
        or step == experiment_config.total_steps
    )
    if should_log:
        append_log_entry(
            step=step,
            drift_matching_loss=float(drift_matching_loss.detach().item()),
            inverse_latent_loss=float(inverse_latent_loss.detach().item()),
            inverse_cycle_loss=float(inverse_cycle_loss.detach().item()),
            drift_norm=drift_norm,
            data_density=data_density,
            model_density=model_density,
            inverse_mode_gap=inverse_mode_gap,
        )

    if (
        step == 1
        or step % experiment_config.snapshot_every_steps == 0
        or step == experiment_config.total_steps
    ):
        snapshot_progress(
            scatter_num_samples=experiment_config.scatter_num_samples,
            show=True,
        )
